# MindEye2 + LoRA — Colab driverRuns the whole experiment: adapt the pretrained shared-subject MindEye2 model to aheld-out subject with ~1 hour of fMRI, using LoRA vs. full fine-tuning vs. a frozenbaseline, then compare them statistically.**Every cell is safe to re-run.** If the runtime disconnects, reconnect and run thecells again from the top — each stage checks the workspace manifest on Drive and skipswork that's already done.Set the runtime to **GPU** (Runtime → Change runtime type → T4 is fine).

## 1. EnvironmentMounts Drive, installs dependencies from the Drive-backed pip cache, and points every library cache at Drive. Fast after the first run.

In [ ]:
import os, sys, subprocessREPO = "/content/mindeye2-lora"if not os.path.exists(REPO):    # replace with your fork if you have one    !git clone https://github.com/<you>/mindeye2-lora.git {REPO}%cd {REPO}from google.colab import drivedrive.mount('/content/drive')# everything persistent lives here — change it if you keep Drive organised differentlyos.environ["MINDEYE_LORA_ROOT"] = "/content/drive/MyDrive/mindeye2_lora"

In [ ]:
!bash setup/colab_setup.sh

In [ ]:
# Colab sometimes needs a restart after installs. If the next cell errors on imports,# run this, then continue from cell 4 (you will NOT lose any downloaded data).# import IPython; IPython.Application.instance().kernel.do_shutdown(True)

## 2. WorkspaceCreates the Drive directory tree and redirects `HF_HOME`, `TORCH_HOME`, and the pip cache into it.

In [ ]:
sys.path.insert(0, f"{REPO}/src")from mindeye_lora.cli import mainmain(["setup", "--config", "configs/colab_t4.yaml"])

## 3. Choose a config| config | runtime | time ||---|---|---|| `configs/smoke.yaml` | any GPU | ~10 min — **start here** || `configs/colab_t4.yaml` | T4 (free) | several hours across sessions || `configs/a100_paper_scale.yaml` | A100/L4 (Pro) | longer; `hidden_dim=4096` |The smoke config runs 3 arms for 10 epochs and exercises every stage. Run it first soconfiguration problems surface in minutes rather than hours.

In [ ]:
CONFIG = "configs/smoke.yaml"      # switch to configs/colab_t4.yaml for the real run

## 4. AssetsDownloads what the experiment needs and nothing else. The 22 GB COCO image file is*sliced remotely over HTTPS* — only the ~1,750 rows this experiment touches are pulled.The 2.86 GB pretrained checkpoint is slimmed to weights-only and the original deleted.First run: 10–20 minutes. Afterwards: instant.

In [ ]:
main(["assets", "--config", CONFIG])

## 5. Verify the pretrained weights loadThis is the guard rail. It builds the model, loads the shared-subject checkpoint, and refuses to continue if any non-ridge parameter is missing — which would silently turn 'fine-tuning' into 'training from scratch'. It also prints which layers LoRA will wrap.

In [ ]:
main(["verify", "--config", CONFIG])

## 6. Precompute CLIP embeddingsEmbeds each stimulus once with OpenCLIP ViT-bigG/14 and caches the 256x1664 token embeddings in fp16. This is what keeps the 2.5 GB vision tower out of memory during training.

In [ ]:
main(["precompute", "--config", CONFIG])

## 7. TrainOne run per (arm, seed). All arms share data order, schedule, and starting weights;only the trainable parameter set differs.State is written to Drive every 10 minutes and at every epoch boundary. `time_budget_min`in the config stops training cleanly before a session is likely to be reclaimed — justre-run this cell in the next session to continue.

In [ ]:
main(["train", "--config", CONFIG])

In [ ]:
# where things stand — run this any time, especially after a disconnectmain(["status", "--config", CONFIG])

## 8. PredictRuns each trained model over the test set and caches the predicted CLIP embeddings (sampled through the diffusion prior).

In [ ]:
main(["predict", "--config", CONFIG])

## 9. ImagesTwo paths, and this cell always produces something.**SDXL unCLIP decoder** (`--decoder sdxl_unclip`): the paper's decoder. 18 GB download,needs Stability's `sgm`, ~3-5 s/image. Frozen and identical across arms, so it adds nobetween-arm variance — every statistic works without it. Worth it on an A100.**Retrieval fallback** (automatic): if the decoder is missing or fails, you get anearest-neighbour panel instead — the closest test-set images to each predictedembedding, top-3, with correct hits outlined. Costs seconds and no downloads.Those are **retrieved photographs, not generated images**. A correct top-1 ispixel-identical to the stimulus, which is a retrieval hit, not a reconstruction. It isalso a coarse discriminator: arms a few percent apart often give identical rows, so readthe statistics tables for the size of any difference.

In [ ]:
# retrieval fallback only (fast, always works)main(["recon", "--config", CONFIG])# real generation — uncomment on an A100 with disk to spare# main(["recon", "--config", CONFIG, "--decoder", "sdxl_unclip", "--n_images", "64"])

## 10. EvaluatePer-image metrics. CLIP-space metrics always; the eight image metrics if reconstructions exist.

In [ ]:
main(["evaluate", "--config", CONFIG])

## 11. ComparePaired statistics against the full fine-tune: bootstrap CIs, Wilcoxon with Holmcorrection, Cohen's d_z, TOST equivalence, and the retention ratio.Expect to see a difference that is *statistically detectable but practically negligible*for a well-chosen rank. Both facts are reported because both are true.

In [ ]:
main(["compare", "--config", CONFIG])

## 12. ReportWrites figures and `REPORT.md` into the Drive workspace under `results/reports/`.

In [ ]:
main(["report", "--config", CONFIG])

In [ ]:
from IPython.display import Markdown, displayfrom pathlib import Pathreport = Path(os.environ["MINDEYE_LORA_ROOT"]) / "results/reports/REPORT.md"display(Markdown(report.read_text()))

In [ ]:
# figures inlinefrom IPython.display import Imagefigdir = Path(os.environ["MINDEYE_LORA_ROOT"]) / "results/figures"for p in sorted(figdir.glob("*.png")):    print(p.name)    display(Image(str(p)))

---## One-shot alternativeEverything above in a single resumable command:```pythonmain(["run-all", "--config", "configs/colab_t4.yaml"])```Re-run it verbatim after any disconnect.## Poking at things```pythonmain(["train", "--config", CONFIG, "--arm", "lora_r16", "--seed", "0"])   # one runmain(["verify", "--config", CONFIG, "--tree"])                            # every Linear layermain(["compare", "--config", CONFIG, "--equivalence_fraction", "0.1"])    # stricter margin```